# 05 — Парсеры полей ценника

Из набора данных от OCR (PaddleOCR) нужно получить:
- штрих-код, артикул, дату печати, скидку
- 2 цены (без карты и по карте)
- доп.инфу (Сухое/Полусладкое/...)
- 12 полей из QR-кода

In [ ]:
import re
import json
from collections import namedtuple

OcrLine = namedtuple('OcrLine', ['text', 'x', 'y', 'w', 'h', 'conf'])


def cx(L): return L.x + L.w / 2
def cy(L): return L.y + L.h / 2

SUPER_TRANS = str.maketrans('⁰¹²³⁴⁵⁶⁷⁸⁹', '0123456789')

## Штрих-код (EAN-13)

На ценнике штрих-код печатается одновременно как полосатый код **и** как 13-значное число под ним. OCR читает цифры. Иногда теряет 1-2 цифры

Стратегия: ищем 10-14 значные цифровые токены, ранжируем их по:
- длине (предпочитается 13),
- позиции (внизу ценника),
- OCR-confidence,

In [ ]:
def ean13_checksum_ok(s):
    s = re.sub(r'\D', '', s)
    if len(s) != 13: return False
    d = [int(c) for c in s]
    parity = sum(v * (3 if i % 2 else 1) for i, v in enumerate(d[:-1]))
    return (10 - parity % 10) % 10 == d[-1]


def parse_barcode(lines):
    if not lines: return None
    H = max(L.y + L.h for L in lines)
    candidates = []
    for L in lines:
        for digits in re.findall(r'\d{10,14}', re.sub(r'\D', '', L.text)):
            if len(digits) > 14: continue
            bottom_bonus = (cy(L) / H) ** 2          # 0..1, выше = ближе ко дну
            valid_bonus = 2.0 if (len(digits) == 13 and ean13_checksum_ok(digits)) else 0.0
            score = len(digits) * 0.5 + bottom_bonus + L.conf + valid_bonus
            candidates.append((digits, score))
    if not candidates: return None
    candidates.sort(key=lambda x: -x[1])
    return candidates[0][0]

In [ ]:

demo_lines = [
    OcrLine('Напиток SANTO STEFANO', 10, 10, 100, 12, 0.9),
    OcrLine('-48%',                  10, 100, 30, 25, 0.95),
    OcrLine('129',                   60, 130, 40, 50, 0.99),  # integer цены
    OcrLine('99',                    100, 130, 20, 20, 0.85), # копейки
    OcrLine('270207736530',          20, 220, 80, 10, 0.7),   # SKU
    OcrLine('4670025474665',         100, 230, 100, 12, 0.85),# barcode
]
print('barcode:', parse_barcode(demo_lines))
print('checksum OK:', ean13_checksum_ok('4670025474665'))

barcode: 4670025474665
checksum OK: True


## Артикул (id_sku)

SKU — 9-12 значное число, печатается **над** штрих-кодом, рядом с датой. Отличаем от barcode по позиции и длине.

In [4]:
def parse_id_sku(lines, barcode_value=None):
    if not lines: return None
    H = max(L.y + L.h for L in lines)
    bc = barcode_value or ''
    candidates = []
    for L in lines:
        for digits in re.findall(r'\d{8,12}', re.sub(r'\D', '', L.text)):
            if not (8 <= len(digits) <= 12): continue
            if bc and (digits == bc or digits in bc or bc in digits): continue
            rel_y = cy(L) / H
            if rel_y < 0.3: continue   # SKU не в верхней трети
            score = L.conf + (1.0 - abs(rel_y - 0.7)) + (0.3 if 9 <= len(digits) <= 12 else 0)
            candidates.append((digits, score))
    if not candidates: return None
    candidates.sort(key=lambda x: -x[1])
    return candidates[0][0]

print('id_sku:', parse_id_sku(demo_lines, parse_barcode(demo_lines)))

id_sku: 270207736530


## Дата печати

Формат `DD.MM.YYYY HH:MM` (например `03.04.2026 3:08`). Иногда OCR разделяет дату и время на две строки — обрабатываем обе ситуации.

In [ ]:
# OCR-substitutions для дат: похожие на цифры буквы → цифры.
# Используем translate(), который работает с int-codepoint ключами.
DIGIT_FIX = str.maketrans({
    'З': '3', 'з': '3',
    'O': '0', 'o': '0', 'О': '0', 'о': '0',
    'I': '1', 'l': '1', 'i': '1',
    'B': '8', 'В': '8',
    'S': '5', 'Ѕ': '5',
})

DATETIME_RE_FULL = re.compile(r'(\d{2})[.\-/](\d{2})[.\-/](\d{4})[\D]+(\d{1,2})[:.]\s*(\d{2})')
DATETIME_RE_DATE = re.compile(r'(\d{2})[.\-/](\d{2})[.\-/](\d{4})')
TIME_RE          = re.compile(r'^(\d{1,2})[:.](\d{2})$')

def parse_print_datetime(lines):
    # 1. Полная запись на одной линии (с OCR-фиксом букв→цифры)
    for L in lines:
        s = L.text.translate(DIGIT_FIX)
        m = DATETIME_RE_FULL.search(s)
        if m:
            dd, mm, yyyy, hh, mi = m.groups()
            return f'{dd}.{mm}.{yyyy} {int(hh)}:{mi}'
    # 2. Только дата на одной линии, время на другой
    for L in lines:
        s = L.text.translate(DIGIT_FIX)
        m = DATETIME_RE_DATE.search(s)
        if m:
            dd, mm, yyyy = m.groups()
            for L2 in lines:
                if L2 is L: continue
                m2 = TIME_RE.match(L2.text.translate(DIGIT_FIX).strip())
                if m2:
                    return f'{dd}.{mm}.{yyyy} {int(m2.group(1))}:{m2.group(2)}'
            return f'{dd}.{mm}.{yyyy}'
    return None

print('datetime full :', parse_print_datetime([OcrLine('03.04.2026 3:08', 0, 200, 80, 10, 0.7)]))
print('datetime fixed:', parse_print_datetime([OcrLine('OЗ.04.2026 З:08', 0, 200, 80, 10, 0.7)]))
print('datetime split:', parse_print_datetime([
    OcrLine('03.04.2026', 0, 200, 80, 10, 0.7),
    OcrLine('3:08',       0, 210, 30, 10, 0.7),
]))

## Размер скидки

На промо-ценнике скидка в круге типа `-48%`. OCR не всегда видит минус, поэтому минус опциональный — но мы **всегда** выводим со знаком.

In [6]:
DISCOUNT_RE = re.compile(r'[-−–—]?\s*(\d{1,2})\s*%')

def parse_discount_amount(lines):
    # Берём наибольший по шрифту (h) — это центральная плашка скидки
    best, best_h = None, 0
    for L in lines:
        m = DISCOUNT_RE.search(L.text)
        if m and 1 <= int(m.group(1)) <= 99 and L.h > best_h:
            best, best_h = int(m.group(1)), L.h
    return f'-{best}%' if best else None

print('discount:', parse_discount_amount(demo_lines))

discount: -48%


## Цены — самый сложный парсер

Особенности:
- На ценнике цена `129⁹⁹` — где `129` это целая часть, `⁹⁹` — superscript копейки. PaddleOCR обычно отдаёт их **двумя отдельными токенами**: `'129'` (большая по шрифту) + `'99'` (маленькая).
- Дефис в `-48%` парсится как integer = 48 → надо **исключать токены с `%`**.
- На красном промо-ценнике почти ВСЕГДА `price_card` оканчивается на `,99` (152 из 154 = 98.7%) — Lenta-конвенция. Используем как override.
- На самом ценнике две цены: `price_default` (зачёркнутая, поменьше) и `price_card` (огромная, активная).

Стратегия:
1. Извлекаем integer-кандидаты из всех токенов (исключая `-N%` строки).
2. Ищем рядом 2-значный токен — это копейки. Если есть — пара `(int, dec)` = price.
3. Если нет — int-only fallback (только для значений ≥ 100).
4. **`price_card`** = кандидат с наибольшим шрифтом (h). **`price_default`** = второй по шрифту.
5. На последнем шаге: для `price_card` форсируем `,99`.

In [7]:
PRICE_INLINE_RE = re.compile(r'(\d{1,5})[,.\s]+(\d{2})\b')

def _extract_integer(text):
    """Тащит integer из строки. Возвращает None если строка — это discount."""
    s = text.translate(SUPER_TRANS).strip()
    # Скип discount-токенов: '48%', '-48%', '-10', и т.д.
    if re.fullmatch(r'[-−–—]?\s*\d{1,2}\s*%?', s):
        if '%' in s or re.fullmatch(r'[-−–—]\s*\d{1,2}', s):
            return None
    runs = re.findall(r'\d+', s)
    if not runs: return None
    longest = max(runs, key=len)
    # Если за longest run сразу идёт '%' — это discount
    idx = s.find(longest)
    if s[idx + len(longest): idx + len(longest) + 1] == '%':
        return None
    v = int(longest)
    return v if 10 <= v <= 99999 else None


def _try_inline_price(text):
    """`129,99` или `129.99` или `129 99` на одной строке. Фильтрует 0.75L и т.п."""
    m = PRICE_INLINE_RE.search(text.translate(SUPER_TRANS))
    if not m: return None
    v = int(m.group(1)) + int(m.group(2)) / 100.0
    return v if v >= 10 else None

In [ ]:
def _candidate_prices(lines):
    """Список (value, anchor_line, source) кандидатов цены."""
    cands = []

    # 1) Inline patterns (целая цена в одной строке: 129,99 / 129.99 / 129 99)
    for L in lines:
        v = _try_inline_price(L.text)
        if v is not None:
            cands.append((v, L, 'inline'))

    # 2) Integer-токены и decimal-токены отдельно
    integers = [(_extract_integer(L.text), L) for L in lines]
    integers = [(v, L) for v, L in integers if v is not None]
    # Декcimal — 2-значный числовой токен. Стрипаем пробелы и не-цифры по краям.
    decimals = []
    for L in lines:
        s = L.text.translate(SUPER_TRANS).strip()
        m = re.fullmatch(r'[^\d]*(\d{2})[^\d]*', s)
        if m:
            decimals.append((int(m.group(1)), L))

    # 3) Пара integer + ближайший 2-значный decimal
    # Расширили окно: dx ≤ 2.5×I.w (раньше 1.5), dy ≤ 1.5×I.h (раньше 1.0)
    for v_int, I in integers:
        best, best_d = None, float('inf')
        for d_val, D in decimals:
            if D is I: continue
            dx = cx(D) - cx(I)
            dy = cy(D) - cy(I)
            if abs(dx) > 2.5 * I.w or abs(dy) > 1.5 * I.h: continue
            # D должен быть СПРАВА (копейки идут после числа)
            if dx < -0.3 * I.w: continue
            d = abs(dx) + 2.0 * abs(dy)   # вертикальная близость важнее
            if d < best_d:
                best, best_d = d_val, d
        if best is not None:
            cands.append((v_int + best / 100.0, I, 'pair'))

    # 4) Int-only fallback (только для значений ≥ 100, чтобы не подобрать `99` копеек как цену)
    for v_int, I in integers:
        if v_int >= 100:
            cands.append((float(v_int), I, 'int-only'))

    return cands

In [ ]:
_SOURCE_PRIORITY = {'inline': 0, 'pair': 1, 'int-only': 2}

def _fmt_price_ru(v):
    return f'{v:.2f}'.replace('.', ',')


def parse_prices(lines):
    """price_card (огромная, ,99) + price_default (зачёркнутая, любые копейки)."""
    out = {'price_default': None, 'price_card': None}
    cands = _candidate_prices(lines)
    if not cands: return out

    best_by_line = {}
    for v, L, src in cands:
        key = id(L)
        if key not in best_by_line or _SOURCE_PRIORITY[src] < _SOURCE_PRIORITY[best_by_line[key][2]]:
            best_by_line[key] = (v, L, src)
    collapsed = list(best_by_line.values())

    # Сортируем по размеру шрифта (h) — крупный шрифт = главная цена
    ordered = sorted(collapsed, key=lambda c: -c[1].h)

    # price_card = самая крупная, форсируем ,99 (Lenta-конвенция)
    if ordered:
        card_v = ordered[0][0]
        out['price_card'] = _fmt_price_ru(int(card_v) + 0.99)

    # price_default = следующая отличающаяся цена (любого source — лучше int-only, чем нет)
    if len(ordered) > 1:
        card_int = int(ordered[0][0])
        for v, L, src in ordered[1:]:
            v_int = int(v)
            if v_int == card_int: continue   # дубликат card
            if abs(v_int - card_int) / max(card_int, 1) <= 0.03: continue
            out['price_default'] = _fmt_price_ru(v)
            break

    return out


print('prices full :', parse_prices(demo_lines))
print('prices fallback (int-only default):', parse_prices([
    OcrLine('129', 0, 0, 50, 80, 0.9),
    OcrLine('99',  60, 0, 20, 30, 0.9),
    OcrLine('252', 200, 0, 30, 30, 0.8),  # default только integer
]))

## Дополнительная информация

На некоторых ценниках — плашка с описанием товара (`Сухое`, `Полусладкое`, `3 по цене 2` и т.д.). Простой keyword-match по списку известных вариантов.

In [ ]:
import difflib

ADDITIONAL_HINTS = (
    'Полусладкое', 'Полусухое', 'Сладкое', 'Сухое',
    'Игристое', 'Шипучее', 'Десертное',
    'по цене', 'Удачная упаковка',
)


OCR_NORM = str.maketrans({
    'о': 'о', 'а': 'а', 'е': 'е', 
})

def _fuzzy_find_hint(text_lower, hint):
    """Ищет hint в text как substring, с допуском Levenshtein-расстояния 1-2."""
    h = hint.lower()
    if h in text_lower:
        return True
    # Скользящее окно длины ≈ len(hint), ищем по нему близкое
    L = len(h)
    if L < 4: return False
    threshold = 0.78 if L >= 8 else 0.83   # для длинных слов чуть мягче
    for i in range(len(text_lower) - L + 1):
        window = text_lower[i:i+L+1]   # +1 для большей толерантности
        ratio = difflib.SequenceMatcher(None, h, window).ratio()
        if ratio >= threshold:
            return True
    return False


def parse_additional_info(lines):
    text_all = ' '.join(L.text for L in lines).lower()
    for hint in ADDITIONAL_HINTS:
        if _fuzzy_find_hint(text_all, hint):
            return hint
    return None

# Тесты
print('exact :', parse_additional_info([OcrLine('Сухое вино', 5, 50, 30, 10, 0.8)]))
print('OCR-err Полусухое→Палусухое:', parse_additional_info([OcrLine('Палусухое', 5, 50, 30, 10, 0.8)]))
print('OCR-err Сухое→Сyое:', parse_additional_info([OcrLine('Сyое', 5, 50, 30, 10, 0.8)]))
print('OCR-err Полусладкое→Полусладко:', parse_additional_info([OcrLine('Полусладко', 5, 50, 30, 10, 0.8)]))
print('no hint:', parse_additional_info([OcrLine('Газировка', 5, 50, 30, 10, 0.8)]))

## Парсер QR-кода

На ценнике есть QR с payload типа:
```
barcode=4690491121887&price1=4421.09&price2=4199.99&price4=2749.99&aP=
```
URL-encoded `key=value&...`. Иногда — JSON `{"b":"...","p1":"..."}`. Пустое значение (`aP=`) = поле отсутствует.

12 ключей маппятся в 12 полей CSV согласно ТЗ.

In [11]:
QR_KEY_MAP = {
    'b': 'qr_code_barcode',                'barcode': 'qr_code_barcode',
    'p1': 'price1_qr',                     'price1': 'price1_qr',
    'p2': 'price2_qr',                     'price2': 'price2_qr',
    'p3': 'price3_qr',                     'price3': 'price3_qr',
    'p4': 'price4_qr',                     'price4': 'price4_qr',
    'wL1C': 'wholesale_level_1_count',     'wholesaleLevel1Count': 'wholesale_level_1_count',
    'wL1P': 'wholesale_level_1_price',     'wholesaleLevel1Price': 'wholesale_level_1_price',
    'wL2C': 'wholesale_level_2_count',     'wholesaleLevel2Count': 'wholesale_level_2_count',
    'wL2P': 'wholesale_level_2_price',     'wholesaleLevel2Price': 'wholesale_level_2_price',
    'aP': 'action_price_qr',               'actionPrice': 'action_price_qr',
    'aC': 'action_code_qr',                'actionCode': 'action_code_qr',
}

ALL_QR_FIELDS = (
    'qr_code_barcode',
    'price1_qr', 'price2_qr', 'price3_qr', 'price4_qr',
    'wholesale_level_1_count', 'wholesale_level_1_price',
    'wholesale_level_2_count', 'wholesale_level_2_price',
    'action_price_qr', 'action_code_qr',
)

def parse_qr_payload(text):
    if not text: return {}
    out = {}
    # Сначала пробуем JSON
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k in QR_KEY_MAP:
                    out[QR_KEY_MAP[k]] = None if v == '' else v
            return out
    except Exception:
        pass
    # Иначе URL-encoded key=value
    for part in re.split(r'[&;,\n]', text):
        m = re.match(r'\s*([A-Za-z0-9_]+)\s*[=:]\s*(.*?)\s*$', part)
        if m and m.group(1) in QR_KEY_MAP:
            out[QR_KEY_MAP[m.group(1)]] = None if m.group(2) == '' else m.group(2)
    return out

demo_qr = 'barcode=4690491121887&price1=4421.09&price2=4199.99&price4=2749.99&aP='
parse_qr_payload(demo_qr)

{'qr_code_barcode': '4690491121887',
 'price1_qr': '4421.09',
 'price2_qr': '4199.99',
 'price4_qr': '2749.99',
 'action_price_qr': None}

## Главный entry-point `parse_all`

Собирает все парсеры в один dict. Используется в pipeline после OCR.

In [12]:
def parse_all(lines):
    barcode = parse_barcode(lines)
    return {
        'barcode': barcode,
        'id_sku': parse_id_sku(lines, barcode),
        'print_datetime': parse_print_datetime(lines),
        'discount_amount': parse_discount_amount(lines),
        'additional_info': parse_additional_info(lines),
        **parse_prices(lines),
    }

# End-to-end demo на реалистичных OCR-линиях
demo_real = [
    OcrLine('Напиток SANTO STEFANO Rosso', 0, 30, 90, 12, 0.85),
    OcrLine('(Россия) 0.25L',              0, 45, 60, 10, 0.7),
    OcrLine('-48%',                        10, 100, 40, 30, 0.9),
    OcrLine('252',                         150, 80, 40, 15, 0.85),   # default price int
    OcrLine('63',                          195, 78, 18, 12, 0.75),   # default copecks (но потом перепишем 99 для card)
    OcrLine('129',                         60, 140, 100, 60, 0.99),  # card price int (большой шрифт)
    OcrLine('99',                          165, 145, 25, 22, 0.85),  # card copecks
    OcrLine('270207736530',                10, 220, 80, 12, 0.7),    # SKU
    OcrLine('03.04.2026 3:08',             10, 235, 100, 10, 0.7),
    OcrLine('Сухое',                       10, 60, 30, 10, 0.6),
    OcrLine('4670025474665',               110, 240, 110, 14, 0.8),  # barcode
]
parse_all(demo_real)

{'barcode': '4670025474665',
 'id_sku': '270207736530',
 'print_datetime': '03.04.2026 3:08',
 'discount_amount': '-48%',
 'additional_info': 'Сухое',
 'price_default': '252,63',
 'price_card': '129,99'}

## Что должно получиться

Для демо выше:
- `barcode`: `'4670025474665'` ✅ (EAN-13 валидный)
- `id_sku`: `'270207736530'`
- `print_datetime`: `'03.04.2026 3:08'`
- `discount_amount`: `'-48%'`
- `additional_info`: `'Сухое'`
- `price_default`: `'252,63'`
- `price_card`: `'129,99'` (`,99` форсируется Lenta-конвенцией)